# 09 · Profile: the industry → job bridge (BLS national matrix vs OEWS New York research estimates)

**Method:** `docs/framework.md` §3. Cards: `docs/source-cards/bls_ind_occ_matrix.md`, `bls_oews_research.md`.

**Why this matters.** brief S4 promises a "way in by industry": pick an industry, see the jobs inside it. The industry sources (QCEW, QWI, nb 07–08) know industries but not jobs; the job sources (projections, wages, nb 02–03) know jobs but not industries. A staffing pattern (what share of an industry's workers are in each job) is the bridge. Two candidates:

| | **BLS National Employment Matrix** | **BLS OEWS research estimates** |
|---|---|---|
| Area | United States | **New York State** |
| Time | 2025 + projected 2035 (one edition) | May 2019 → May 2025 (seven years) |
| Industry detail | BLS EP industry codes (NAICS-based, to 6 digits) | NAICS sector → 6 digit |
| Covers | All workers incl. self-employed | Wage and salary jobs only |
| Wages | No | Yes, by industry × job |

Neither is NYC. This notebook profiles both, then runs a test: apply each one's staffing pattern to NYC's own industry employment (QCEW, nb 07) and compare the result with NYC's own job counts (NYS DOL wages, nb 03). The bridge that rebuilds NYC's job mix better is the one to use.

**It changes nothing.**

In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import openpyxl
import pandas as pd
from IPython.display import display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from common import RAW

pd.set_option("display.max_columns", 40, "display.max_colwidth", 70, "display.width", 220)
SOC = r"\d\d-\d{4}"
man = pd.read_csv(RAW / "manifest.csv", dtype=str, keep_default_na=False)
display(man[man.key.isin(["bls_ind_occ_matrix", "bls_oews_research"])][["key", "file", "downloaded_utc", "bytes", "note"]])

,key,file,downloaded_utc,bytes,note
148,bls_ind_occ_matrix,jobs/bls_ind_occ_matrix/matrix_2026-10-12.xlsx,2026-10-12T17:26:26+00:00,7933317,"National employment matrix: jobs within each industry, 2024 + 2034..."
149,bls_oews_research,jobs/bls_oews_research/oes_research_2019_allsectors.xlsx,2026-10-12T17:26:29+00:00,132639549,OEWS research estimates May 2019: state x industry x job (NY = 36)
150,bls_oews_research,jobs/bls_oews_research/oes_research_2020_allsectors.xlsx,2026-10-12T17:26:32+00:00,136892691,OEWS research estimates May 2020: state x industry x job (NY = 36)
151,bls_oews_research,jobs/bls_oews_research/oes_research_2021_allsectors.xlsx,2026-10-12T17:26:35+00:00,135740097,OEWS research estimates May 2021: state x industry x job (NY = 36)
152,bls_oews_research,jobs/bls_oews_research/oes_research_2022_allsectors.xlsx,2026-10-12T17:26:38+00:00,139350964,OEWS research estimates May 2022: state x industry x job (NY = 36)
153,bls_oews_research,jobs/bls_oews_research/oes_research_2023_allsectors.xlsx,2026-10-12T17:26:41+00:00,146131550,OEWS research estimates May 2023: state x industry x job (NY = 36)
154,bls_oews_research,jobs/bls_oews_research/oes_research_2024_allsectors.xlsx,2026-10-12T17:26:44+00:00,155080180,OEWS research estimates May 2024: state x industry x job (NY = 36)
158,bls_oews_research,jobs/bls_oews_research/oes_research_2025_allsectors.xlsx,2026-10-13T01:45:47+00:00,149024500,OEWS research estimates May 2025: state x industry x job (NY = 36)


---
# Part A · BLS National Employment Matrix (2025–35)

## A0. Source header

| | **National Employment Matrix** (`bls_ind_occ_matrix`) |
|---|---|
| **Publisher** | BLS Employment Projections (EP) |
| **File** | `matrix.xlsx`: the 2025–35 matrix (same edition as `education.xlsx` and `occupation.xlsx`, nb 04 and D1) ([EP matrix page](https://www.bls.gov/emp/data/occupational-data.htm)) |
| **How it was collected** | **Model / compilation**: wage and salary jobs by industry × job from OEWS (national), industry totals from CES and other sources, self-employed from CPS; then projected to 2035 by BLS economists ([EP methods](https://www.bls.gov/emp/documentation/projections-methods.htm)) |
| **Units** | Employment in thousands; percents of industry and of occupation |
| **License** | Public domain |

In [2]:
MATRIX = RAW / man.loc[man.key == "bls_ind_occ_matrix", "file"].iloc[0]
m = pd.read_excel(MATRIX, sheet_name="Matrix", dtype=str, keep_default_na=False)
print(f"{len(m):,} rows x {m.shape[1]} cols"); print(list(m.columns))
print("Text cells with stray spaces:", {c: int((m[c] != m[c].str.strip()).sum()) for c in m.columns if (m[c] != m[c].str.strip()).any()} or "none")
NUMS = [c for c in m.columns if c[:4] in ("2025", "2035")]
print("Non-numeric cells in numeric columns:", {c: m.loc[~m[c].str.fullmatch(r"-?\d+(\.\d+)?"), c].value_counts().head(3).to_dict() for c in NUMS if (~m[c].str.fullmatch(r"-?\d+(\.\d+)?")).any()} or "none")
display(m.head(3))

113,898 rows x 14 cols
['Occupation type', 'Industry type', 'Occupation code', 'Occupation title', 'Industry code', 'Industry title', '2025 Employment', '2025 Percent of Industry', '2025 Percent of Occupation', '2035 Employment', '2035 Percent of Industry', '2035 Percent of Occupation', 'Numeric change, 2025-35', 'Percent change, 2025-35']


Text cells with stray spaces: none
Non-numeric cells in numeric columns: none


,Occupation type,Industry type,Occupation code,Occupation title,Industry code,Industry title,2025 Employment,2025 Percent of Industry,2025 Percent of Occupation,2035 Employment,2035 Percent of Industry,2035 Percent of Occupation,"Numeric change, 2025-35","Percent change, 2025-35"
0,Summary,Summary,00-0000,"Total, all occupations",TE1000,"Total, all industries",170280.8,100,100,176198.5,100,100,5917.6,3.5
1,Summary,Summary,11-0000,Management occupations,TE1000,"Total, all industries",13683.4,8,100,14527.6,8.2,100,844.2,6.2
2,Summary,Summary,11-1000,Top executives,TE1000,"Total, all industries",3904.1,2.3,100,4095.2,2.3,100,191.1,4.9


**Load findings**
- `Matrix` sheet: 113,898 rows × 14 columns, one edition (2025–35, the same as `education.xlsx` and `occupation.xlsx`). No stray spaces, no symbols in the numeric columns: every cell is a plain number.
- The manifest note said "2024 + 2034"; that was the old edition's wording. Fixed in `src/sources.py` (the file itself is 2025–35).

## A2–A4. Grain, codes and hierarchy

In [3]:
print("Duplicates on occupation code + industry code:", m.duplicated(["Occupation code", "Industry code"]).sum())
ind = m[["Industry code", "Industry title", "Industry type"]].drop_duplicates()
occ = m[["Occupation code", "Occupation title", "Occupation type"]].drop_duplicates()
print(f"Industries: {len(ind)} ({ind['Industry type'].value_counts().to_dict()}) | occupations: {len(occ)} ({occ['Occupation type'].value_counts().to_dict()})")
print("Industry code patterns:", ind["Industry code"].str.replace(r"\d", "9", regex=True).value_counts().to_dict())
print("Non-numeric industry codes:"); display(ind[~ind["Industry code"].str.fullmatch(r"\d{6}")])
print("Occupation codes that are line items but end in 0 (BLS aggregates, not SOC detailed):")
agg = occ[(occ["Occupation type"] == "Line item") & occ["Occupation code"].str.endswith("0")]
display(agg)

Duplicates on occupation code + industry code: 0


Industries: 423 ({'Line item': 292, 'Summary': 131}) | occupations: 1112 ({'Line item': 831, 'Summary': 281})
Industry code patterns: {'999999': 355, '9999A9': 15, '99999P': 10, '99999L': 10, '99999S': 9, '9999-9': 6, '9999P9': 4, '9999Y9': 4, 'TE9999': 3, '99-999': 3, '999-99': 3, '9999R9': 1}
Non-numeric industry codes:


,Industry code,Industry title,Industry type
0,TE1000,"Total, all industries",Summary
1112,TE1100,Self-employed workers,Line item
2004,TE1200,Total wage and salary employment,Summary
3824,1131-2,Forestry,Line item
13954,31-330,Manufacturing,Summary
...,...,...,...
95446,6242-3,"Community food and housing, emergency and other relief services, a...",Summary
98314,7113-4,"Promoters of performing arts and sports, and agents for public fig...",Summary
107075,8132-3,Grantmaking and giving services and social advocacy organizations,Summary
108149,8134-9,"Civic, social, professional, and similar organizations",Summary


Occupation codes that are line items but end in 0 (BLS aggregates, not SOC detailed):


,Occupation code,Occupation title,Occupation type
52,13-1020,Buyers and purchasing agents,Line item
74,13-2020,Property appraisers and assessors,Line item
463,29-2010,Clinical laboratory technologists and technicians,Line item
497,31-1120,Home health and personal care aides,Line item
634,39-7010,Tour and travel guides,Line item
819,47-4090,Miscellaneous construction and related workers,Line item
917,51-2090,Miscellaneous assemblers and fabricators,Line item


**Grain findings**
- Key `Occupation code + Industry code` is unique .
- **423 industries** (292 line items + 131 summaries) × 1,112 occupations (831 line items + 281 summaries). Only cells with jobs are listed.
- Industry codes are not plain NAICS: 68 of 423 have letters or dashes. `TE1000` all industries, `TE1100` self-employed, `31-330` manufacturing, and suffixes for ownership: `P` private, `S` state, `L` local, `Y` state + local (e.g. `61110L` local public schools).
- **7 BLS aggregate job codes** are line items (`13-1020`, `13-2020`, `29-2010`, `31-1120`, `39-7010`, `47-4090`, `51-2090`), same as in nb 04/06.

## A6. Relationships: do line items add up, and does the matrix match Table 1.2 (nb 04/D1)?

In [4]:
mm = m.copy()
for c in NUMS: mm[c] = pd.to_numeric(mm[c], errors="coerce")
tot = mm[(mm["Occupation code"] == "00-0000")].set_index("Industry code")["2025 Employment"]
li = mm[mm["Occupation type"] == "Line item"].groupby("Industry code")["2025 Employment"].sum()
diff = (li / tot - 1).mul(100)
print(f"Line-item jobs vs the industry's total, % difference: median {diff.median():.2f}, max |diff| {diff.abs().max():.2f} over {diff.notna().sum()} industries")
# Do line-item industries add up to the all-industry total, per occupation?
all_ind = mm[mm["Industry code"] == "TE1000"].set_index("Occupation code")["2025 Employment"]
li_ind = mm[mm["Industry type"] == "Line item"].groupby("Occupation code")["2025 Employment"].sum()
d2 = (li_ind / all_ind - 1).mul(100).dropna()
print(f"Line-item industries vs all industries, per occupation: median {d2.median():.2f}%, max |diff| {d2.abs().max():.2f}%")
# Why: line-item industries are not a complete split of their sector. Compare each sector row with its line items
ii = mm[mm["Occupation code"] == "00-0000"][["Industry code", "Industry title", "Industry type", "2025 Employment"]].copy()
ii["s2"] = ii["Industry code"].str[:2].replace({"32": "31", "33": "31", "45": "44", "49": "48"})
sec_rows = ii[ii["Industry code"].str.fullmatch(r"\d\d0000|\d\d-\d\d0")].set_index("s2")["2025 Employment"]
li_rows = ii[ii["Industry type"] == "Line item"].groupby("s2")["2025 Employment"].sum()
cov_li = pd.concat([sec_rows.rename("sector_row"), li_rows.rename("sum_line_items")], axis=1)
cov_li["line_items_pct"] = (cov_li.sum_line_items / cov_li.sector_row * 100).round(1)
print("Sector row vs the sum of its line-item industries (thousands, 2025):"); display(cov_li)

OCC12 = RAW / man.loc[man.key == "bls_ep_occupation", "file"].iloc[0]
t12 = pd.read_excel(OCC12, sheet_name="Table 1.2", dtype=str, header=1, keep_default_na=False)
t12 = t12[t12.iloc[:, 1].str.fullmatch(SOC)]
e12 = pd.to_numeric(t12.set_index(t12.columns[1])[t12.columns[3]], errors="coerce")
j = pd.concat([all_ind.rename("matrix"), e12.rename("table_1_2")], axis=1, join="inner")
print(f"\nTable 1.2 column used: {t12.columns[3]!r} | codes in both: {len(j)} | exact match: {(j.matrix.round(1) == j.table_1_2.round(1)).mean():.1%}")
print(f"Total, all occupations 2025: matrix {all_ind['00-0000']:,.1f}k | Table 1.2 {e12.get('00-0000', float('nan')):,.1f}k")

Line-item jobs vs the industry's total, % difference: median -1.07, max |diff| 72.41 over 422 industries


Line-item industries vs all industries, per occupation: median -6.67%, max |diff| 96.29%
Sector row vs the sum of its line-item industries (thousands, 2025):


,sector_row,sum_line_items,line_items_pct
s2,,,
11,1462.4,1462.4,100.0
21,573.5,573.5,100.0
22,601.3,339.6,56.5
23,8266.7,7177.2,86.8
31,12633.7,12366.9,97.9
42,6059.5,5394.9,89.0
44,15440.9,14864.8,96.3
48,6639.5,7234.9,109.0
51,2861.9,2477.4,86.6



Table 1.2 column used: 'Employment, 2025' | codes in both: 1112 | exact match: 100.0%
Total, all occupations 2025: matrix 170,280.8k | Table 1.2 170,280.8k


**Relationship findings**
- Within an industry, line-item jobs add to the industry total (median −1.1%).
- The matrix total ties to Table 1.2 exactly: 1,112 of 1,112 codes, 170,280.8k jobs in 2025.
- But line-item industries are not a full split of their sector. Utilities' line items cover 57% of the sector row, food services and accommodation 49%, arts 66%, other services 70%, admin/support 79%. The bridge has to use the sector rows, not add up line items.
- Transportation's line items are 109% of its sector row: the Postal Service (`491000`, 595.5k) sits outside the `48-490` row.
- Government: sector rows `610000` and `620000` include public schools and hospitals ("state, local, and private"); `999100/999200/999300` are government excluding education, hospitals and postal.

---
# Part B · OEWS research estimates, New York State (May 2019 → May 2025)

## B0. Source header

| | OEWS Research Estimates by State and Industry (`bls_oews_research`) |
|---|---|
| **Publisher** | BLS Occupational Employment and Wage Statistics (OEWS) |
| **Files** | `oes_research_YYYY_allsectors.xlsx`, 2019–2025 (all states; we keep NY = `36`) ([OEWS research estimates](https://www.bls.gov/oes/oes-research-estimates.htm)) |
| **How it was collected** | The OEWS employer survey (3-year pooled sample), tabulated by state × industry. BLS calls them research estimates: they're not official OEWS products and are less reliable than the published tables ("use with caution") |
| **Covers** | Wage and salary jobs; no self-employed. Includes government (NAICS 99) |
| **Symbols** | `**` employment n/a, `*` wage n/a, `#` top-coded wage, `~` percent < 0.05% |
| **License** | Public domain |

In [5]:
def read_ny(path):
    # Stream the sheet and keep New York rows only; every cell kept as text (framework rule)
    t = time.time()
    wb = openpyxl.load_workbook(path, read_only=True)
    ws = wb[wb.sheetnames[0]]  # the data sheet is first; its name changes by year
    it = ws.iter_rows(values_only=True)
    head = [str(h).strip().lower().replace(" ", "_") for h in next(it)]  # 2019 says "occ code", 2020+ "OCC_CODE"
    ai = head.index("area")
    rows, n = [], 0
    for r in it:
        n += 1
        if str(r[ai]) == "36":
            rows.append(["" if v is None else str(v) for v in r])
    d = pd.DataFrame(rows, columns=head)
    print(f"{path.name}: {n:,} rows, {len(d):,} NY rows, {time.time() - t:.0f}s")
    return d

R = {}
for f in sorted(man.loc[man.key == "bls_oews_research", "file"]):
    y = f.split("_")[-2]
    R[y] = read_ny(RAW / f)
base = set(R["2021"].columns)
print("Columns that differ from 2021 (after lower-case + underscores):", {y: sorted(set(d.columns) ^ base) for y, d in R.items() if set(d.columns) != base})
print("Column order differs by year too (2019 puts i_group after o_group), so code always selects by name.")

oes_research_2019_allsectors.xlsx: 836,739 rows, 29,349 NY rows, 64s


oes_research_2020_allsectors.xlsx: 816,060 rows, 28,274 NY rows, 69s


oes_research_2021_allsectors.xlsx: 818,972 rows, 28,335 NY rows, 68s


oes_research_2022_allsectors.xlsx: 835,717 rows, 28,792 NY rows, 70s


oes_research_2023_allsectors.xlsx: 855,397 rows, 29,505 NY rows, 70s


oes_research_2024_allsectors.xlsx: 861,922 rows, 29,887 NY rows, 76s


oes_research_2025_allsectors.xlsx: 867,905 rows, 30,244 NY rows, 71s
Columns that differ from 2021 (after lower-case + underscores): {'2024': ['naics2', 'none', 'release']}
Column order differs by year too (2019 puts i_group after o_group), so code always selects by name.


**Load findings**
- **Seven files**, May 2019 → May 2025 (May 2025 was posted 2026-05-15 and downloaded in this notebook). 816k–868k rows each for all states; 28,274–30,244 New York rows. Each file takes ~70–80 s to stream (~9 min for all seven).
- **Headers change by year**: 2019 is lower-case with spaces (`occ code`, `occ title`), 2020+ upper-case with underscores, 2024 adds `naics2`, `release` and an empty column. Column order differs too (2019 puts `i_group` after `o_group`), and the sheet name changes every year. Read by name after normalising headers (lower-case, spaces → `_`).

## B2–B4. Grain, codes and special values

In [6]:
for y, d in R.items():
    k = d.duplicated(["naics", "i_group", "occ_code"]).sum()
    k2 = d.duplicated(["naics", "occ_code"]).sum()
    sym = {c: d[c].isin(["**", "*", "#", "~"]).sum() for c in ["tot_emp", "pct_total", "a_median"]}
    hyb = d.occ_code.isin(["15-1256", "13-1198", "13-2098", "15-1245"]).sum()
    print(f"{y}: dup naics+i_group+occ {k} (naics+occ alone {k2}) | i_group {d.i_group.value_counts().to_dict()} | o_group {d.o_group.value_counts().to_dict()} | symbols {sym} | hybrid SOC codes {hyb}")
r24 = R["2024"]
t = r24[r24.o_group == "total"]
print("\nSame NAICS code at two levels (a 3-digit industry with one 4-digit child, listed twice with identical numbers), 2024:")
display(t[t.naics.duplicated(keep=False)][["naics", "naics_title", "i_group", "tot_emp"]])
print("NAICS 2017 → 2022 shows up as code changes, e.g. general merchandise 452000 (2019–21) → 455000 (2022+):",
      {y: sorted(set(d.naics) & {"452000", "455000"}) for y, d in R.items()})
print("\n2024 sector codes:"); display(r24[r24.i_group == "sector"][["naics", "naics_title"]].drop_duplicates())

2019: dup naics+i_group+occ 0 (naics+occ alone 463) | i_group {'4-digit': 14049, '3-digit': 8221, 'sector': 4349, '5-digit': 2539, '6-digit': 191} | o_group {'detailed': 24360, 'major': 4585, 'total': 404} | symbols {'tot_emp': np.int64(2859), 'pct_total': np.int64(6010), 'a_median': np.int64(515)} | hybrid SOC codes 559
2020: dup naics+i_group+occ 0 (naics+occ alone 452) | i_group {'4-digit': 13409, '3-digit': 7941, 'sector': 4228, '5-digit': 2496, '6-digit': 200} | o_group {'detailed': 23341, 'major': 4526, 'total': 407} | symbols {'tot_emp': np.int64(3016), 'pct_total': np.int64(5779), 'a_median': np.int64(585)} | hybrid SOC codes 569
2021: dup naics+i_group+occ 0 (naics+occ alone 489) | i_group {'4-digit': 13402, '3-digit': 7989, 'sector': 4286, '5-digit': 2494, '6-digit': 164} | o_group {'detailed': 23426, 'major': 4501, 'total': 408} | symbols {'tot_emp': np.int64(3248), 'pct_total': np.int64(3366), 'a_median': np.int64(666)} | hybrid SOC codes 0
2022: dup naics+i_group+occ 0 (na

,naics,naics_title,i_group,tot_emp
4084,327000,Nonmetallic Mineral Product Manufacturing,3-digit,15080
4085,327000,Nonmetallic Mineral Product Manufacturing,4-digit,15080
9021,455000,General Merchandise Retailers,3-digit,137160
9022,455000,General Merchandise Retailers,4-digit,137160
10272,484000,Truck Transportation,3-digit,38650
10273,484000,Truck Transportation,4-digit,38650
12414,517000,Telecommunications,3-digit,35040
12415,517000,Telecommunications,4-digit,35040
13259,523000,"Securities, Commodity Contracts, and Other Financial Investments a...",3-digit,215370
13260,523000,"Securities, Commodity Contracts, and Other Financial Investments a...",4-digit,215370


NAICS 2017 → 2022 shows up as code changes, e.g. general merchandise 452000 (2019–21) → 455000 (2022+): {'2019': ['452000'], '2020': ['452000'], '2021': ['452000'], '2022': ['455000'], '2023': ['455000'], '2024': ['455000'], '2025': ['455000']}

2024 sector codes:


,naics,naics_title
0,11,"Agriculture, Forestry, Fishing and Hunting"
97,21,"Mining, Quarrying, and Oil and Gas Extraction"
221,22,Utilities
612,23,Construction
2908,31-33,Manufacturing
6482,42,Wholesale Trade
8688,44-45,Retail Trade
10835,48-49,Transportation and Warehousing
11362,51,Information
12872,52,Finance and Insurance


**Grain findings**
- Key `naics + i_group + occ_code` is unique in all seven years. `naics + occ_code` alone is not (450–514 duplicates a year): a 3-digit industry with only one 4-digit child is listed twice with identical numbers (`327000`, `455000`, `484000`, `517000`, `523000`, `531000`).
- Levels: sector, 3-, 4-, 5-, 6-digit NAICS; jobs at detailed and major group level plus a total row.
- **Codes change over time.** 2019–20 use OEWS hybrid SOC codes (559 and 569 rows), 2021+ pure SOC 2018. NAICS 2017 in 2019–21, NAICS 2022 from 2022 (general merchandise `452000` → `455000`).
- Symbols: 2,490–3,623 `**` (employment not published) per year; `~` in `pct_total` (under 0.05%); `#` top-coded wages.
- Sector `99`'s own title says it all: "Federal, State, and Local Government, excluding State and Local Government Schools and Hospitals and the U.S. Postal Service". Public schools and hospitals are inside sectors 61 and 62, and the Postal Service inside `48-49`.

## B5. Coverage: how much of each industry do the detailed jobs account for?

Within an industry, the `pct_total` of detailed jobs should add to ~100%. A gap is jobs whose cell wasn't published.

In [7]:
def coverage(d):
    x = d[d.o_group == "detailed"].copy()
    x["pct"] = pd.to_numeric(x.pct_total.replace("~", "0.02"), errors="coerce")
    x["emp"] = pd.to_numeric(x.tot_emp, errors="coerce")
    tot = pd.to_numeric(d[d.o_group == "total"].set_index(["naics", "i_group"]).tot_emp, errors="coerce")
    g = x.groupby(["naics", "i_group"]).agg(pct_sum=("pct", "sum"), emp_sum=("emp", "sum"))
    g["level"] = g.index.get_level_values("i_group")
    g["emp_share"] = g.emp_sum / tot.reindex(g.index) * 100
    return g
cov = {y: coverage(d) for y, d in R.items()}
print("Median % of an industry's jobs covered by published detailed jobs (by employment), per level and year:")
display(pd.DataFrame({y: c.groupby("level").emp_share.median().round(1) for y, c in cov.items()}))
print("2024 sectors:"); display(cov["2024"][cov["2024"].level == "sector"].round(1))

Median % of an industry's jobs covered by published detailed jobs (by employment), per level and year:


,2019,2020,2021,2022,2023,2024,2025
level,,,,,,,
3-digit,93.9,93.3,94.0,95.0,94.9,93.9,94.5
4-digit,88.6,88.2,87.4,89.4,89.0,87.2,88.5
5-digit,89.7,89.4,88.4,91.5,91.6,90.2,89.2
6-digit,80.0,75.2,74.7,78.1,81.5,70.3,71.8
sector,98.2,97.5,97.7,97.9,97.9,97.5,97.0


2024 sectors:


,,pct_sum,emp_sum,level,emp_share
naics,i_group,,,,
11,sector,79.8,2590.0,sector,79.4
21,sector,81.2,3670.0,sector,80.7
22,sector,92.7,36240.0,sector,92.6
23,sector,97.3,382650.0,sector,97.3
31-33,sector,99.3,409770.0,sector,99.3
42,sector,97.5,299590.0,sector,97.6
44-45,sector,99.0,834210.0,sector,99.0
48-49,sector,98.1,349350.0,sector,98.1
51,sector,97.5,263990.0,sector,97.5


**Coverage findings**
- At the sector level, published detailed jobs cover 97–98% of each industry's employment (median, every year). Deeper levels lose more: 3-digit ~94%, 4-digit ~88%, 6-digit 70–80%.
- Weakest sectors in 2024: agriculture (79%) and mining (81%), both tiny in NYC. Every sector that matters for NYC is at 93–99.6%.
- Use the sector level for the bridge, renormalising shares to the published detail. Deeper levels only for drill-down, with a "share not published" remainder.

---
# Part C · Which bridge rebuilds NYC's job mix better?

**Test.** For each sector *s* and job *j*, take the share of the sector's jobs in *j* from each bridge. Multiply by NYC's 2024 jobs in that sector and add across sectors: that's an *estimated* NYC job count by occupation. Compare it with NYC's measured job counts (NYS DOL occupational wages, nb 03, NYC region). Score = the dissimilarity index: half the sum of absolute share differences (0 = identical job mix, 100 = no overlap).

Government has to be filed the same way on both sides. Both bridges put public schools and hospitals inside sectors 61 and 62, next to private ones, and keep a "government" pattern for everything else public (police, transit, agencies). The OEWS research page says so for the research files ([May 2025 page](https://www.bls.gov/oes/oes_research_2025.htm)); the matrix shows it in its titles (`610000` "state, local, and private", `999300` "local government, excluding education and hospitals"). One difference: the matrix keeps the Postal Service (`491000`) outside its `48-49` row, the research file inside.

So NYC's jobs must be split the same way. NYS DOL's QCEW (nb 07) gives NYC government only as one total (566,579), so this test uses BLS QCEW county files (`bls_qcew_county`, new in this notebook), which publish government by sector for each borough, and fills their suppressed cells from NYS DOL where it can.

In [8]:
# NYC 2024 jobs by sector and ownership from BLS QCEW county files (agglvl 74 = county x ownership x sector)
qb = pd.concat([pd.read_csv(RAW / f, dtype=str) for f in man.loc[man.key == "bls_qcew_county", "file"] if f.split("/")[-1].startswith("2024")])
qb["emp"] = qb.annual_avg_emplvl.astype(int)
OWN = {"1": "federal", "2": "state", "3": "local", "5": "private"}
own_tot = qb[qb.agglvl_code == "71"].groupby("own_code").emp.sum().rename(OWN)
sec = qb[qb.agglvl_code == "74"].pivot_table(index="industry_code", columns="own_code", values="emp", aggfunc="sum", fill_value=0).rename(columns=OWN)
supp = qb[(qb.agglvl_code == "74") & (qb.disclosure_code == "N") & qb.own_code.isin(["1", "2", "3"])]
print("NYC 2024 jobs by ownership (county totals):", own_tot.to_dict(), "| all:", f"{own_tot.sum():,}")
print("Published sector cells vs ownership totals (the gap is suppressed cells):", (own_tot - sec.sum()).to_dict())
print(f"Suppressed government sector cells: {len(supp)}"); display(supp.groupby(["own_code", "industry_code"]).area_fips.apply(list).rename(index=OWN, level=0).to_frame("boroughs"))
print("NYC government jobs by sector, published cells:"); display(sec.loc[sec[["federal", "state", "local"]].sum(axis=1) > 0, ["federal", "state", "local"]])

NYC 2024 jobs by ownership (county totals): {'federal': 45969, 'state': 34796, 'local': 485816, 'private': 4064226} | all: 4,630,807
Published sector cells vs ownership totals (the gap is suppressed cells): {'federal': 1, 'state': 14995, 'local': 129337, 'private': 14053}
Suppressed government sector cells: 29


boroughs
own_code industry_code                                     
state    22                                         [36081]
         52                                         [36061]
         54                                         [36061]
         61                    [36005, 36047, 36061, 36085]
         62             [36005, 36047, 36061, 36081, 36085]
local    23                                         [36081]
         48-49                        [36005, 36047, 36061]
         52                                         [36061]
         53                                  [36061, 36085]
         54                                         [36061]
         56             [36005, 36047, 36061, 36081, 36085]
         61                                  [36081, 36085]
         62                                  [36081, 36085]

NYC government jobs by sector, published cells:


own_code,federal,state,local
industry_code,,,
44-45,61,0,0
48-49,16320,0,2796
51,8,0,0
52,276,0,0
53,0,0,3561
54,445,0,0
61,3,0,142051
62,5385,0,36259
71,715,0,0


NYC government by sector (BLS QCEW county files)
- 35 files (5 boroughs × 2019–2025), annual averages, by ownership × NAICS. They tie to NYS DOL's NYC QCEW within ±2 jobs for each level of government and +608 private jobs (0.015%) in 2024.
- **They publish government by sector**, which NYS DOL's NYC file mostly doesn't: NYC local government 2024 has 142,051 jobs in education and 36,259 in health care in published cells, and 171,812 in public administration.
- 129,337 local jobs are in suppressed (`N`) cells: Manhattan/Bronx/Brooklyn transit, Queens and Staten Island schools and health, and a few small cells. NYS DOL fills transit (55,174 NYC local transportation jobs). The 76,959 still unplaced are tested both ways below (as schools, or as other government). State education and health come from NYS DOL (2,477 and 11,661).

In [9]:
# Fill the gaps. NYS DOL's NYC-level QCEW (nb 07) publishes some cells BLS suppresses by borough:
QCEW = RAW / man.loc[man.key == "nysdol_qcew_annual", "file"].iloc[0]
qc = pd.read_csv(QCEW, dtype=str, keep_default_na=False); qc.columns = [c.strip() for c in qc.columns]
qc = qc[(qc.Area == "New York City") & (qc["Area Type"] == "Labor Market Area") & (qc.Year == "2024")]
dol = lambda naics, own: int(qc[(qc.NAICS == naics) & (qc.Ownership == own)]["Average Employment"].iloc[0])
LBL = {"federal": "Federal Government", "state": "State Government", "local": "Local Government", "private": "Private"}
print("Tie-out, BLS county files minus NYS DOL NYC (2024 jobs):", {k: int(own_tot[k] - dol("00", v)) for k, v in LBL.items()})
transit_local = dol("48", "Local Government")                         # MTA / NYC Transit, local government
state_61, state_62 = dol("61", "State Government"), dol("62", "State Government")
print(f"NYS DOL fills: local transportation {transit_local:,} | state education {state_61:,} | state health {state_62:,}")

# Public jobs inside 61/62 (schools, colleges, hospitals) vs the rest of government
pub_61 = sec.loc["61", "federal"] + state_61 + sec.loc["61", "local"]
pub_62 = sec.loc["62", "federal"] + state_62 + sec.loc["62", "local"]
local_left = own_tot["local"] - sec["local"].sum() - (transit_local - sec.loc["48-49", "local"])
print(f"Local government jobs still unplaced after the transit fill: {local_left:,} "
      "(suppressed cells: mostly Queens and Staten Island schools, plus Queens/SI health and small cells)")
usps = sec.loc["48-49", "federal"]

def nyc_jobs(left_to, postal_own_row):
    s = sec["private"].drop("99").copy()          # private 'unclassified' (99) has no staffing pattern: dropped
    s["61"] += pub_61 + (local_left if left_to == "61" else 0)
    s["62"] += pub_62
    gov = own_tot[["federal", "state", "local"]].sum() - pub_61 - pub_62 - (local_left if left_to == "61" else 0)
    if postal_own_row:
        s["491"] = usps; gov -= usps                 # matrix: Postal Service is its own row
    else:
        s["48-49"] += usps; gov -= usps              # research: Postal Service sits inside 48-49
    s["99"] = gov
    return s

V = pd.DataFrame({"research, left→education": nyc_jobs("61", False), "research, left→government": nyc_jobs("99", False),
                  "matrix, left→education": nyc_jobs("61", True), "matrix, left→government": nyc_jobs("99", True)}).fillna(0).astype(int)
print("NYC 2024 jobs used per sector (the two ways of placing the unplaced local jobs):"); display(V.loc[["48-49", "491", "61", "62", "99"]])
print("Column totals:", V.sum().to_dict())

Tie-out, BLS county files minus NYS DOL NYC (2024 jobs): {'federal': 0, 'state': 0, 'local': 2, 'private': 608}
NYS DOL fills: local transportation 55,174 | state education 2,477 | state health 11,661
Local government jobs still unplaced after the transit fill: 76,959 (suppressed cells: mostly Queens and Staten Island schools, plus Queens/SI health and small cells)
NYC 2024 jobs used per sector (the two ways of placing the unplaced local jobs):


,"research, left→education","research, left→government","matrix, left→education","matrix, left→government"
industry_code,,,,
48-49,178652,178652,162332,162332
491,0,0,16320,16320
61,418751,341792,418751,341792
62,1037784,1037784,1037784,1037784
99,275466,352425,275466,352425


Column totals: {'research, left→education': 4595271, 'research, left→government': 4595271, 'matrix, left→education': 4595271, 'matrix, left→government': 4595271}


In [10]:
W = RAW / man.loc[man.key == "nysdol_oews_regions", "file"].iloc[0]
w = pd.read_csv(W, dtype=str, keep_default_na=False); w.columns = [c.strip() for c in w.columns]
w = w[(w["Area Name"].str.strip() == "New York City") & w["Standard Occupational Code"].str.fullmatch(SOC) & ~w["Standard Occupational Code"].str.endswith("0")]
actual = pd.to_numeric(w.set_index("Standard Occupational Code").Employment, errors="coerce").dropna()
print(f"NYC measured jobs (NYS DOL, detailed codes with employment): {actual.sum():,.0f} across {len(actual)} jobs")

def research_shares(d):
    x = d[(d.i_group == "sector") & (d.o_group == "detailed")].copy()
    x["emp"] = pd.to_numeric(x.tot_emp, errors="coerce")
    x["share"] = x.emp / x.groupby("naics").emp.transform("sum")   # renormalised to the published detail
    return x.rename(columns={"naics": "sector", "occ_code": "occ"})[["sector", "occ", "share"]]

# National matrix: sector rows (public schools and hospitals included), government and postal line items
SECT_M = {"110000": "11", "210000": "21", "220000": "22", "230000": "23", "31-330": "31-33", "420000": "42", "44-450": "44-45",
          "48-490": "48-49", "510000": "51", "520000": "52", "530000": "53", "540000": "54", "550000": "55", "560000": "56",
          "610000": "61", "620000": "62", "710000": "71", "720000": "72", "810000": "81",
          "491000": "491", "999100": "99", "999200": "99", "999300": "99"}
print("Matrix codes missing:", [k for k in SECT_M if k not in set(mm["Industry code"])])
mx = mm[(mm["Occupation type"] == "Line item") & mm["Industry code"].isin(SECT_M)].copy()
mx["sector"] = mx["Industry code"].map(SECT_M)
mx = mx.groupby(["sector", "Occupation code"], as_index=False)["2025 Employment"].sum().rename(columns={"Occupation code": "occ"})
mx["share"] = mx["2025 Employment"] / mx.groupby("sector")["2025 Employment"].transform("sum")

def rebuild(sh, jobs):
    return sh.assign(nyc=sh.share * sh.sector.map(jobs)).groupby("occ").nyc.sum()

def dissimilarity(est, act):
    idx = act.index.union(est.index)
    a = act.reindex(idx, fill_value=0); e = est.reindex(idx, fill_value=0)
    return 50 * (a / a.sum() - e / e.sum()).abs().sum()

res, ests = [], {}
for y in sorted(R):
    if y < "2021": continue  # 2019-20 use OEWS hybrid codes, not SOC 2018
    sh = research_shares(R[y])
    for left in ["education", "government"]:
        e = rebuild(sh, V[f"research, left→{left}"]); ests[(f"research {y}", left)] = e
        res.append({"bridge": f"OEWS research NY {y}", "unplaced local jobs →": left, "dissimilarity": dissimilarity(e, actual)})
for left in ["education", "government"]:
    e = rebuild(mx, V[f"matrix, left→{left}"]); ests[("matrix", left)] = e
    res.append({"bridge": "National matrix 2025", "unplaced local jobs →": left, "dissimilarity": dissimilarity(e, actual)})
print("Dissimilarity with NYC's measured job mix (lower = better):")
display(pd.DataFrame(res).pivot(index="bridge", columns="unplaced local jobs →", values="dissimilarity").round(1))

NYC measured jobs (NYS DOL, detailed codes with employment): 4,047,620 across 556 jobs


Matrix codes missing: []
Dissimilarity with NYC's measured job mix (lower = better):


unplaced local jobs →,education,government
bridge,,
National matrix 2025,23.0,23.4
OEWS research NY 2021,17.7,17.9
OEWS research NY 2022,16.5,16.8
OEWS research NY 2023,16.2,16.6
OEWS research NY 2024,15.2,15.4
OEWS research NY 2025,13.9,14.2


In [11]:
best = sorted(R)[-1]
cmp_ = pd.DataFrame({"NYC measured": actual, f"est. NY research {best}": ests[(f"research {best}", "education")].round(-1),
                     "est. national": ests[("matrix", "education")].round(-1)}).dropna(subset=["NYC measured"])
cmp_["title"] = w.set_index("Standard Occupational Code")["Occupational Title"].str.strip()
print("Largest NYC jobs: measured vs rebuilt from industry employment (unplaced local jobs → education)")
display(cmp_.sort_values("NYC measured", ascending=False).head(15))
cmp_["err_ny"] = cmp_[f"est. NY research {best}"] - cmp_["NYC measured"]
print(f"Biggest misses (NY research {best} bridge):")
display(cmp_.reindex(cmp_.err_ny.abs().sort_values(ascending=False).index).head(12).drop(columns="err_ny"))

Largest NYC jobs: measured vs rebuilt from industry employment (unplaced local jobs → education)


,NYC measured,est. NY research 2025,est. national,title
11-1021,99410.0,102020.0,102930.0,General and Operations Managers
37-2011,96390.0,95940.0,64900.0,"Janitors and Cleaners, Except Maids and Housekeeping Cleaners"
41-2031,95100.0,86130.0,77230.0,Retail Salespersons
29-1141,89560.0,106670.0,140100.0,Registered Nurses
33-9032,87040.0,65330.0,36360.0,Security Guards
15-1252,74710.0,69770.0,73510.0,Software Developers
35-3031,70150.0,66130.0,58070.0,Waiters and Waitresses
35-3023,66090.0,76300.0,95940.0,Fast Food and Counter Workers
23-1011,63670.0,53910.0,28310.0,Lawyers
43-9061,63530.0,64000.0,75050.0,"Office Clerks, General"


Biggest misses (NY research 2025 bridge):


,NYC measured,est. NY research 2025,est. national,title
33-9032,87040.0,65330.0,36360.0,Security Guards
29-1141,89560.0,106670.0,140100.0,Registered Nurses
43-4051,58860.0,73630.0,88530.0,Customer Service Representatives
37-3011,8770.0,22420.0,26820.0,Landscaping and Groundskeeping Workers
43-6014,49430.0,60420.0,56030.0,"Secretaries and Administrative Assistants, Except Legal, Medical, ..."
53-3032,16340.0,26850.0,44860.0,Heavy and Tractor-Trailer Truck Drivers
35-3023,66090.0,76300.0,95940.0,Fast Food and Counter Workers
13-2051,41810.0,31660.0,15420.0,Financial Analysts
23-1011,63670.0,53910.0,28310.0,Lawyers
25-9045,50780.0,60540.0,43460.0,"Teaching Assistants, Except Postsecondary"


Test findings: the New York research estimates win clearly

| Bridge | Dissimilarity vs NYC's measured job mix |
|---|---|
| National matrix 2025 | **23.0** |
| OEWS research NY 2021 → 2024 | 17.7 → 16.5 → 16.2 → 15.2 |
| **OEWS research NY 2025** | **13.9** |

- Where the 76,959 unplaced local government jobs go barely matters: ±0.2–0.4 points.
- An earlier version of this test was unfair (NY research 16.6 vs matrix 23.0). It put all of NYC's government jobs, public schools included, on the "government" pattern, which in both bridges *excludes* schools. That made too few elementary teachers and too many correctional officers and police. With schools and hospitals filed inside 61/62 on both sides, those misses are gone.
- Why the newer years score better: NYS DOL's NYC counts come from the same survey cycle as the May 2025 research file (OEWS panels Nov 2022 → May 2025). Caveat: the two aren't independent. NYC is a big part of New York State's OEWS sample, so the NY research file is partly built from the same NYC responses. That's a reason to prefer it for NYC, but it also flatters the score.
- **What's left over** (NY research 2025): NYC has more security guards (87k measured vs 65k rebuilt), lawyers and financial analysts, and fewer RNs (90k vs 107k), customer service reps, truck drivers and landscapers than New York State's staffing patterns predict. These are real NYC vs rest-of-state differences inside a sector (Manhattan offices vs upstate). The national matrix misses all of them by more: it rebuilds only 36k security guards and 28k lawyers, and puts 140k RNs in NYC.

## Part D · Stability over time (NY research): how much does a sector's job mix move between years?

In [12]:
def shares(d):
    return research_shares(d).set_index(["sector", "occ"]).share
pairs = [("2021", "2024"), ("2024", sorted(R)[-1])]
out = {}
for a, b in pairs:
    j = pd.concat([shares(R[a]).rename("a"), shares(R[b]).rename("b")], axis=1).fillna(0)
    out[f"{a}→{b}"] = 50 * (j.a - j.b).abs().groupby(level=0).sum()
print("Dissimilarity between years in each sector's job mix (SOC 2018 in all years used):")
display(pd.DataFrame(out).round(1).sort_values(f"2021→2024", ascending=False))

Dissimilarity between years in each sector's job mix (SOC 2018 in all years used):


,2021→2024,2024→2025
sector,,
21,22.0,17.5
11,18.6,16.8
71,18.1,9.0
22,16.0,7.8
51,15.3,6.9
81,15.1,9.1
56,14.9,8.4
55,14.3,12.6
53,13.6,6.7


**Stability findings**
- A sector's job mix moves 8–22 points from 2021 to 2024 and 4–9 points in a single year (2024 → 2025) for the big NYC sectors (health 5.0, education 4.5, retail 4.9, finance 7.3). Small sectors (mining, agriculture) move most, and part of that is sampling noise.
- So a staffing pattern is fine for "jobs inside this industry *now*", but use the latest year, and don't present year-to-year changes in a pattern as real hiring shifts.

---
## Fitness for purpose

| Question | National matrix | NY research | Why |
|---|---|---|---|
| S4: "way in by industry": the jobs inside an NYC industry, now | fallback | Use | NY research rebuilds NYC's job mix far better (13.9 vs 23.0), has wages, and is updated yearly |
| Jobs inside an industry in 2035 | Use (shares only) | no | Only the matrix projects. Apply its 2025 → 2035 *change in share* on top of the NY pattern, labeled "national trend" |
| Self-employed workers by job | yes | no | Only the matrix has `TE1100`. Context only |
| Hires by job (QWI hires × staffing pattern) | partial | estimate | Possible as an estimate: it assumes hires follow the staffing pattern, which undercounts high-turnover jobs. Label it so, or don't show it |
| Wages by industry × job | no | state level | NY State, not NYC. NYC wages stay from NYS DOL (nb 03) |

**Recommendation for v1:** use the OEWS research estimates for New York, latest year (May 2025), at the NAICS sector level, as the industry → job bridge. The national matrix stays for the 2035 view and self-employment. Label the bridge "estimated from New York State staffing patterns (BLS research estimates)" everywhere it shows.

## Issues list

| # | Issue | Source | Proposed treatment (decided in staging) |
|---|---|---|---|
| 1 | Headers, column order and sheet names change by year | research | Normalise headers (lower-case, `_`), select by name |
| 2 | Same NAICS code at two levels | research | Key includes `i_group`; bridge uses sectors only |
| 3 | Hybrid SOC (2019–20) and NAICS 2017 → 2022 | research | Bridge uses 2021+; keep `soc_vintage`, `naics_vintage` columns |
| 4 | `**`, `~`, `#` symbols | research | `NULL` + flag columns; `~` → `pct_lt_0_05 = true` |
| 5 | Detailed jobs cover 97–98% of a sector | research | Renormalise shares within sector; keep the covered % |
| 6 | "Research estimates", not official | research | Label on every visual that uses the bridge |
| 7 | Public schools and hospitals inside 61/62; Postal inside 48-49 (research) or its own row (matrix) | both | NYC sector jobs built the same way (BLS QCEW county + NYS DOL fills) |
| 8 | Line-item industries don't add up to sectors | matrix | Use sector rows only |
| 9 | Industry codes with letters (`P`/`S`/`L`/`Y`, `TE`, `31-330`) | matrix | Own industry code column; map sector rows to NAICS sectors |
| 10 | 7 BLS aggregate job codes | matrix | Job dimension keeps them (nb 06) |
| 11 | 129k NYC local government jobs suppressed by borough × sector | QCEW county | Fill transit from NYS DOL; put the rest in a labeled "unplaced" bucket (test shows it barely matters) |

## Proposed staging tables

**`stg_bls__oews_research_ny`**
- **Grain / key:** `(year, naics, i_group, occ_code)`, New York State only
- **Columns:** `year`, `naics`, `naics_title`, `i_group`, `occ_code`, `occ_title`, `o_group`, `tot_emp`, `emp_prse`, `pct_total`, `h_median`, `a_median`, `a_mean`, `emp_suppressed`, `pct_lt_0_05`, `wage_topcoded`, `soc_vintage` (`hybrid` / `2018`), `naics_vintage` (`2017` / `2022`)
- **Filters:** area `36`, 2019 → 2025 (204,386 rows)

**`stg_bls__ind_occ_matrix`**
- **Grain / key:** `(industry_code, occ_code)`
- **Columns:** `industry_code`, `industry_title`, `industry_type`, `occ_code`, `occ_title`, `occ_type`, `emp_2025`, `pct_of_industry_2025`, `pct_of_occ_2025`, `emp_2035`, `pct_of_industry_2035`, `pct_of_occ_2035`, `naics_sector` (sector rows only)
- **Filters:** none (113,898 rows)

**`stg_bls__qcew_county`**
- **Grain / key:** `(county_fips, year, own_code, industry_code)`, annual
- **Columns:** `county_fips`, `year`, `own_code`, `industry_code`, `agglvl_code`, `estabs`, `emp`, `total_wages`, `avg_annual_pay`, `disclosed`
- **Filters:** agglvl 71 (ownership totals) and 74 (ownership × sector), 2019 → 2025 (1,365 rows). Used only for the government split; NYS DOL QCEW (nb 07) stays the main industry source